# 🧠 01B — Entrenamiento Expertos 3D

Entrena los dos expertos 3D del proyecto:
- **Experto 4** → LUNA16 CT 3D (`R3D-18`)
- **Experto 5** → Pancreatic Cancer CT 3D (`R3D-18` baseline estable)

Incluye: FP16, Gradient Accumulation, Gradient Checkpointing y Early Stopping.


## 0. Setup

In [2]:
!pip install nibabel openpyxl --quiet


In [3]:
import sys, torch
from pathlib import Path

sys.path.insert(0, '/workspace/moe_medical_vision/src')

from data.datasets import get_dataloader
from losses import FocalLoss
from models.experts_3d import build_luna_expert, build_pancreatic_expert
from train.train_3d import seed_everything, sanity_check_single_batch, fit_3d_expert

seed_everything(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CHECKPOINT_DIR = Path('/workspace/moe_medical_vision/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LUNA_PATH = '/workspace/moe_medical_vision/data/raw/luna16'
PANCREATIC_PATH = '/workspace/moe_medical_vision/data/raw/pancreatic'

print('device:', DEVICE)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('gpu:', props.name, round(props.total_memory/1024**3, 2), 'GB')


device: cuda
gpu: NVIDIA GeForce RTX 4090 23.53 GB


## 1. Sanity check — LUNA16 loader + model

In [4]:
train_loader_luna, train_ds_luna = get_dataloader('luna16', LUNA_PATH, split='train', batch_size=2, num_workers=2)
val_loader_luna, val_ds_luna = get_dataloader('luna16', LUNA_PATH, split='val', batch_size=2, num_workers=2)

model_luna = build_luna_expert(pretrained=True, use_gradient_checkpointing=True).to(DEVICE)
criterion_luna = torch.nn.CrossEntropyLoss()

info_luna = sanity_check_single_batch(model_luna, train_loader_luna, criterion_luna, DEVICE)
print(info_luna)


[LUNA16 total] 888 | nódulo:601 | sano:287
[LUNA16 train] 711 volúmenes
[INFO] Batch size → 2 (3D)
[LUNA16 total] 888 | nódulo:601 | sano:287
[LUNA16 val] 177 volúmenes
[INFO] Batch size → 2 (3D)


Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth
100%|██████████| 127M/127M [00:00<00:00, 205MB/s]  


{'input_shape': (2, 1, 64, 64, 64), 'logits_shape': (2, 2), 'loss': 0.5573766827583313, 'labels': [1, 1]}


## 2. Entrenar Experto 4 — LUNA16

In [ ]:
optimizer_luna = torch.optim.AdamW(model_luna.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_luna = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_luna, mode='min', factor=0.5, patience=2)

luna_result = fit_3d_expert(
    model=model_luna,
    train_loader=train_loader_luna,
    val_loader=val_loader_luna,
    criterion=criterion_luna,
    optimizer=optimizer_luna,
    scheduler=scheduler_luna,
    device=DEVICE,
    epochs=15,
    accum_steps=4,
    mixed_precision=True,
    patience=5,
    checkpoint_path=CHECKPOINT_DIR / 'expert4_luna16_r3d18_best.pth',
)
luna_result


/workspace/moe_medical_vision/src/train/train_3d.py:138: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=mixed_precision and device.startswith("cuda"))
/workspace/moe_medical_vision/src/train/train_3d.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=mixed_precision and device.startswith("cuda")):
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


## 3. Sanity check — Pancreatic loader + model

In [ ]:
train_loader_pan, train_ds_pan = get_dataloader('pancreatic', PANCREATIC_PATH, split='train', batch_size=1, num_workers=2)
val_loader_pan, val_ds_pan = get_dataloader('pancreatic', PANCREATIC_PATH, split='val', batch_size=1, num_workers=2)

model_pan = build_pancreatic_expert(pretrained=True, use_gradient_checkpointing=True).to(DEVICE)
alpha_pan = train_ds_pan.get_focal_alpha()
criterion_pan = FocalLoss(gamma=2.0, alpha=alpha_pan)

info_pan = sanity_check_single_batch(model_pan, train_loader_pan, criterion_pan, DEVICE)
print('alpha_pan:', alpha_pan)
print(info_pan)


## 4. Entrenar Experto 5 — Pancreatic

In [ ]:
optimizer_pan = torch.optim.AdamW(model_pan.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_pan = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_pan, mode='min', factor=0.5, patience=2)

pan_result = fit_3d_expert(
    model=model_pan,
    train_loader=train_loader_pan,
    val_loader=val_loader_pan,
    criterion=criterion_pan,
    optimizer=optimizer_pan,
    scheduler=scheduler_pan,
    device=DEVICE,
    epochs=20,
    accum_steps=8,
    mixed_precision=True,
    patience=6,
    checkpoint_path=CHECKPOINT_DIR / 'expert5_pancreatic_r3d18_best.pth',
)
pan_result


## 5. Resumen

In [ ]:
for f in sorted(CHECKPOINT_DIR.glob('expert*_best.pth')):
    print(f.name, round(f.stat().st_size / 1e6, 1), 'MB')

print('Siguiente paso recomendado: 02_backbone_y_routers.ipynb SOLO si ambos expertos 3D quedaron validados.')
